In [2]:
import numpy as np
import scipy.stats as st
import math

### Datos Agrupados (Media y Cuasivarianza)
Úsalo cuando te den una tabla de frecuencias con intervalos (como en el Plástico A de la Pregunta 3).

In [3]:
# 1. Ingresa las marcas de clase (punto medio del intervalo) y sus frecuencias
marcas_clase = np.array([147, 153, 159, 165])
frecuencias = np.array([5, 12, 16, 8])

# 2. Cálculos automáticos
n = np.sum(frecuencias)
media_agrupada = np.sum(marcas_clase * frecuencias) / n

# Cuasivarianza (dividido en n-1)
suma_cuadrados = np.sum(frecuencias * (marcas_clase**2))
varianza_agrupada = (suma_cuadrados - n * (media_agrupada**2)) / (n - 1)
desviacion_agrupada = np.sqrt(varianza_agrupada)

print("--- Estadísticas de Datos Agrupados ---")
print(f"Tamaño de muestra (n): {n}")
print(f"Media: {media_agrupada:.3f}")
print(f"Cuasivarianza (S^2): {varianza_agrupada:.3f}")
print(f"Desviación Estándar (S): {desviacion_agrupada:.3f}")

--- Estadísticas de Datos Agrupados ---
Tamaño de muestra (n): 41
Media: 156.951
Cuasivarianza (S^2): 31.698
Desviación Estándar (S): 5.630


### Intervalo de Confianza para Razón de Varianzas (Prueba F)
Úsalo para saber si dos poblaciones tienen varianzas "iguales" o "homogéneas" antes de comparar sus medias. Si el intervalo contiene al 1, son estadísticamente iguales.

In [4]:
# 1. Datos de las dos muestras (A y B)
var_A = 31.698  # Varianza S^2 de A
n_A = 41        # Tamaño muestra A

var_B = 27.04   # Varianza S^2 de B
n_B = 25        # Tamaño muestra B

nivel_confianza = 0.95

# 2. Cálculos
alfa = 1 - nivel_confianza
df_A = n_A - 1
df_B = n_B - 1
ratio_varianzas = var_A / var_B

# Valores críticos de la distribución F
# OJO: scipy usa (df_numerador, df_denominador)
f_inf = st.f.ppf(alfa/2, df_A, df_B)
f_sup = st.f.ppf(1 - alfa/2, df_A, df_B)

limite_inferior_F = ratio_varianzas / f_sup
limite_superior_F = ratio_varianzas / f_inf

print("--- IC para Cuociente de Varianzas ---")
print(f"Ratio Muestral (S^2_A / S^2_B): {ratio_varianzas:.3f}")
print(f"Intervalo al {nivel_confianza*100}%: [{limite_inferior_F:.3f} ; {limite_superior_F:.3f}]")
if limite_inferior_F <= 1 <= limite_superior_F:
    print("Conclusión: Las varianzas son homogéneas (el IC contiene al 1).")
else:
    print("Conclusión: Las varianzas son distintas (el IC NO contiene al 1).")

--- IC para Cuociente de Varianzas ---
Ratio Muestral (S^2_A / S^2_B): 1.172
Intervalo al 95.0%: [0.546 ; 2.353]
Conclusión: Las varianzas son homogéneas (el IC contiene al 1).


### Intervalo de Confianza para Diferencia de Medias (Varianzas Iguales)
Úsalo después del Bloque 3, si determinaste que las varianzas son homogéneas. Usa la distribución t de Student y la "Varianza Combinada" ($S_p^2$).

In [6]:
# 1. Datos de las dos muestras
media_A = 156.951
media_B = 154.0

# Reutilizamos datos del bloque anterior
# var_A = 31.698, n_A = 41
# var_B = 27.04, n_B = 25

nivel_confianza_t = 0.99
alfa_t = 1 - nivel_confianza_t
df_total = n_A + n_B - 2

# 2. Varianza combinada (Sp^2)
sp_cuadrado = ((n_A - 1)*var_A + (n_B - 1)*var_B) / df_total
sp = np.sqrt(sp_cuadrado)

# 3. Diferencia de medias y Error Estándar
diferencia_medias = media_A - media_B
error_estandar_diff = sp * np.sqrt((1/n_A) + (1/n_B))

# Valor t crítico
t_critico = st.t.ppf(1 - alfa_t/2, df_total)

margen_error_diff = t_critico * error_estandar_diff
limite_inf_t = diferencia_medias - margen_error_diff
limite_sup_t = diferencia_medias + margen_error_diff

print("--- IC para Diferencia de Medias (Varianzas Homogéneas) ---")
print(f"Varianza Combinada (Sp^2): {sp_cuadrado:.3f}")
print(f"Diferencia puntual: {diferencia_medias:.3f}")
print(f"Valor t crítico: {t_critico:.3f}")
print(f"Intervalo al {nivel_confianza_t*100}%: [{limite_inf_t:.3f} ; {limite_sup_t:.3f}]")

--- IC para Diferencia de Medias (Varianzas Homogéneas) ---
Varianza Combinada (Sp^2): 29.951
Diferencia puntual: 2.951
Valor t crítico: 2.655
Intervalo al 99.0%: [-0.736 ; 6.638]


 ### Evaluación de un Estimador Máximo Verosímil (EMV)
 El desarrollo algebraico (derivar e igualar a 0) se hace a mano, pero una vez que despejas el estimador $\hat{\theta}$, usas Python para evaluar rápidamente la sumatoria de la muestra sin equivocarte en la calculadora, como en la Pregunta 1.

In [5]:
# 1. Ingresa la muestra de la Pregunta 1
muestra_emv = np.array([3, 1, 7, 3, 8, 11, 7, 1, 8, 4])
n_emv = len(muestra_emv)

# 2. El estimador despejado fue: beta = sqrt(n / sum(X_i^2))
suma_cuadrados_X = np.sum(muestra_emv**2)
beta_estimado = np.sqrt(n_emv / suma_cuadrados_X)

# 3. Invarianza: Probabilidad P(X > 8)
# La fórmula teórica era e^(-64 * beta^2)
prob_mayor_8 = np.exp(-64 * (beta_estimado**2))

print("--- Cálculo Numérico de EMV ---")
print(f"Sumatoria de X_i al cuadrado: {suma_cuadrados_X}")
print(f"Estimación de Beta: {beta_estimado:.4f}")
print(f"Probabilidad estimada P(X > 8): {prob_mayor_8:.4f} ({prob_mayor_8*100:.2f}%)")

--- Cálculo Numérico de EMV ---
Sumatoria de X_i al cuadrado: 383
Estimación de Beta: 0.1616
Probabilidad estimada P(X > 8): 0.1881 (18.81%)
